# Code Generation Evaluation Notebook

In [1]:
!pip install -q datasets requests torch peft bitsandbytes transformers trl accelerate sentencepiece wandb matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.0 MB/s eta 0:00:00


## 1. Core Imports and Configuration

In [2]:
import math
import multiprocessing
import json
import re
import os
import time
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import torch

NUM_SAMPLES = 1 # Number of samples to generate per problem
K_VALUE = 1 # K for pass@k metric
RESULTS_DIR = "data"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Mount Drive
# drive.mount('/content/drive')

# HF Login
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## 2. Prompt Templates

In [3]:
ZERO_SHOT_PROMPT = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Problem:
{prompt}

Code:
"""

FEW_SHOT_PROMPT = """You are an expert Python programmer. You will be provided some examples, and then a problem to solve.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Example 1:
Problem:
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"

Code:
```python
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"
    return a + b
```

Example 2:
Problem:
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"

Code:
```python
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"
    return s == s[::-1]
```

Problem:
{prompt}

Code:
"""

COT_PROMPT = """You are an expert Python programmer. Solve the following problem by thinking step-by-step.
First, provide a brief explanation of your logic, then provide the code block.

Problem:
{prompt}

Solution:
"""

# Dictionary to manage all templates
PROMPT_TEMPLATES = {
    "zero_shot": ZERO_SHOT_PROMPT,
    "few_shot": FEW_SHOT_PROMPT,
    # "cot": COT_PROMPT,
}


## 3. Helper Functions

In [4]:
def run_eval(code, test_code, queue):
    local_scope = {}
    try:
        exec(code, local_scope)
        exec(test_code, local_scope)
        queue.put(True)
    except Exception:
        queue.put(False)

def exec_code(code, test_code, timeout=5):
    """Executes the code and the test, returning True if passes, False otherwise using multiprocessing"""
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=run_eval, args=(code, test_code, queue))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return False

    return queue.get() if not queue.empty() else False

def calculate_pass_at_k(n, c, k):
    """
    Calculate pass@k metric.
    n: total samples
    c: correct samples
    k: k in pass@k
    """
    if n - c < k:
        return 1.0

    if c == 0:
        return 0.0
    try:
        return 1.0 - (math.comb(n - c, k) / math.comb(n, k))
    except ValueError:
        return 0.0

def clean_code(generated_text):
    # Regex to extract code between ```python and ```
    match = re.search(r"```python\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)

    # Fallback to general block
    match = re.search(r"```\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)

    return generated_text.strip()

def save_results(results, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

## 4. Data Loading

In [5]:
dataset = load_dataset("openai_humaneval", split="test")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Dataset({
    features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
    num_rows: 164
})


## 5. Model Initialization

In [6]:
from transformers import BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"

# Config 4-bit quantization for GPU VRAM optimization (saves ~10GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

Memory footprint: 5.4 GB


## 6. Model Generator Class

In [7]:
class QwenHFGenerator:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate(self, prompt_template, task_prompt, num_samples=1):
        full_prompt = prompt_template.format(prompt=task_prompt)

        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)

        samples = []
        for _ in range(num_samples):
            try:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.2,
                    top_p=0.95,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )

                prompt_length = inputs["input_ids"].shape[1]
                generated_ids = outputs[0][prompt_length:]
                generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
                samples.append(generated_text)
            except Exception as e:
                print(f"Error during generation: {e}")
                samples.append("")

        return samples

generator = QwenHFGenerator(base_model, tokenizer)

## 7. Evaluation Loop

In [8]:
for template_name, prompt_template in PROMPT_TEMPLATES.items():
    print(f"\n===== Evaluating Template: {template_name} =====")
    results = []
    total_pass_at_k = 0

    for idx, item in enumerate(tqdm(dataset)):
        task_id = item["task_id"]
        prompt = item["prompt"]
        test_cases = item["test"]
        entry_point = item["entry_point"]

        # 1. Generate code
        start_time = time.time()
        raw_samples = generator.generate(prompt_template, prompt, num_samples=NUM_SAMPLES)
        time_taken = time.time() - start_time

        # 2. Extract and Evaluate
        correct_count = 0
        cleaned_codes = []

        for raw_text in raw_samples:
            code = clean_code(raw_text)
            eval_code = code + "\n" + test_cases + f"\ncheck({entry_point})\n"

            passed = exec_code(code, eval_code, timeout=5)
            if passed:
                correct_count += 1

            cleaned_codes.append({
                "raw": raw_text,
                "extracted": code,
                "passed": passed
            })

        pass_at_k = calculate_pass_at_k(NUM_SAMPLES, correct_count, K_VALUE)
        total_pass_at_k += pass_at_k

        results.append({
            "task_id": task_id,
            "task_name": entry_point,
            "samples": cleaned_codes,
            "correct_count": correct_count,
            "n": NUM_SAMPLES,
            "pass@k": pass_at_k,
            "time_taken": time_taken
        })

    avg_pass_at_k = total_pass_at_k / len(dataset)
    avg_time = sum(r["time_taken"] for r in results) / len(results)
    print(f"Template [{template_name}] - Avg pass@{K_VALUE}: {avg_pass_at_k:.4f} - Avg time: {avg_time:.2f}s/it")

    # Save separate results for each template
    filename = os.path.join(RESULTS_DIR, f"results_{template_name}.json")
    save_results(results, filename)
    print(f"Results saved to {filename}")


===== Evaluating Template: zero_shot =====


100%|██████████| 164/164 [30:03<00:00, 11.00s/it]


Template [zero_shot] - Avg pass@1: 0.7622 - Avg time: 10.92s/it
Results saved to data/results_zero_shot.json

===== Evaluating Template: few_shot =====


100%|██████████| 164/164 [40:01<00:00, 14.64s/it]

Template [few_shot] - Avg pass@1: 0.7866 - Avg time: 14.56s/it
Results saved to data/results_few_shot.json
